# Fusion de flux multi-modèle sur séries brutes (yeux + tête)

Même architecture de stacking que `Fusion_multi-models.ipynb` mais les modèles de flux
reçoivent des séries temporelles brutes rééchantillonnées par minute (3D : n_segments × T × n_canaux)
au lieu de features agrégées.

**Source** : `DataPhase1_time.csv` (~70 Hz) segmenté en fenêtres de T pas de temps par minute.

In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pipeline.extract import load_per_minute_targets
from pipeline.pretraitement import apply_target_discretization
from pipeline.raw_segmentation import (
    prepare_splits_and_impute_3d,
    segment_sequences_by_minute,
    segment_sequences_sliding_window,
)
from pipeline.evaluation import (
    evaluate_by_subject,
    evaluate_robustness,
    evaluate_test_set,
    plot_pca_if_classification,
)
from pipeline.fusion import (
    build_meta_model,
    make_meta_features,
    split_feature_streams,
    train_stream_models,
)
from pipeline.reporting import export_visual_report

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Seed globale: {SEED}")

## 1) Profils modulaires

In [ ]:
DATA_PROFILE = {
    "source": "csv",
    "file_path": r"../data/Données_brutes/DataPhase1_time.csv",
    "subject_id_col": "Participant",
    "time_col": "Time",
}

PREPROCESS_PROFILE = {
    "imputation_strategy": "median",
    "normalization": "standard",
}

TARGET_PROFILE = {
    "source": "xlsx",
    "xlsx_path": r"../data/Questionnaires/FMS1_org.xlsx",
    "sheet_name": "Feuil1",
    "subject_id_col": "Sujet",
    "target_mode": "per_minute",
    "minute_columns": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    "clip_quantiles": None,
    "discretize": {
        "bins":   [0, 5, 10, 20],
        "labels": ["low", "medium", "high"],
    },
}

MODEL_PROFILE = {
    "model_type": "cnn_1d",
    "task_type": "classification",
    "split_method": "group",
    "test_size": 0.20,
    "val_size": 0.20,
    "class_weight": "balanced",
    "random_state": SEED,
}

# None : m\u00eame mod\u00e8le pour les deux flux
# Ou : {"eye": {**MODEL_PROFILE, "model_type": "cnn_1d"}, "head": {**MODEL_PROFILE, "model_type": "bilstm"}}
STREAM_PROFILES = None

# Canaux bruts assignés \u00e0 chaque flux (noms de colonnes exacts de DataPhase1_time.csv)
FUSION_PROFILE = {
    "streams": {
        "eye": [
            "X gaze direction",
            "Y gaze direction",
            "Left Pupil Diameter",
            "Right Pupil Diameter",
        ],
        "head": [
            "RotX",
            "RotY",
            "RotZ",
            "HMDPosX",
            "HMDPosY",
            "HMDPosZ",
            "isBoat",
            "X World Position",
            "Y World Position",
        ],
    },
    "meta_model": "logistic_regression",
}

RAW_PROFILE = {
    "sequence_length": 60,  # T : pas de temps par minute (1 Hz équivalent)
    "pad_value": 0.0,
}

EVAL_PROFILE = {
    "robustness_noise_std": 0.01,
    "robustness_repeats": 5,
}

OUTPUT_PROFILE = {
    "output_dir": r"../data/outputs/fusion_raw",
    "save_visual_report": True,
    "visual_report_format": "both",
    "visual_report_name": "visual_report_raw",
    "visual_report_functions": [
        "visual_hypothesis_page",
        "visual_cover_page",
        "visual_model_architecture_page",
        "visual_split_report",
        "visual_metrics_table_page",
        "visual_confusion_matrix_page",
        "visual_missing_values_bar",
        "visual_correlation_pages",
        "visual_violin_pages",
    ],
    "max_corr_features": 13,
    "max_violin_features": 13,
    "violin_features_per_page": 6,
    "hypothesis": "Fusion brute : CNN-1D x2 + meta LR (baseline raw)",
}

_active_profiles = STREAM_PROFILES if STREAM_PROFILES is not None else MODEL_PROFILE
_model_type_display = (
    {k: v["model_type"] for k, v in STREAM_PROFILES.items()}
    if STREAM_PROFILES is not None
    else MODEL_PROFILE["model_type"]
)
print("Profils chargés")
print("Mod\u00e8le flux    :", _model_type_display)
print("T\u00e2che          :", MODEL_PROFILE["task_type"])
print("Méta-mod\u00e8le    :", FUSION_PROFILE["meta_model"])
print("T (pas/min)    :", RAW_PROFILE["sequence_length"])
print("Normalisation  :", PREPROCESS_PROFILE.get("normalization") or "aucune")

## 3) Chargement et segmentation des données brutes

In [ ]:
from pipeline.extract import load_features_for_approach
from pipeline.pretraitement import apply_column_aggregations

print("Chargement des donnees brutes...")
raw_df = load_features_for_approach(DATA_PROFILE, PREPROCESS_PROFILE)
# raw_df a maintenant subject_id et time (renommes par load_csv_features)

# Agregations de colonnes (ex: Pupil diameter AVG)
raw_df = apply_column_aggregations(raw_df, PREPROCESS_PROFILE)

print(f"Shape : {raw_df.shape}")
print(f"Sujets : {raw_df['subject_id'].nunique()}")
print(f"Colonnes : {list(raw_df.columns[:10])}...")

# Feature cols = union de tous les canaux declares dans les flux
all_stream_cols = []
for cols in FUSION_PROFILE["streams"].values():
    for c in cols:
        if c not in all_stream_cols:
            all_stream_cols.append(c)

missing = [c for c in all_stream_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f"Colonnes absentes dans le CSV brut : {missing}")

feature_cols = all_stream_cols
print("Canaux retenus (" + str(len(feature_cols)) + ") :", feature_cols)


## 4) Segmentation par minute et jointure cible

In [ ]:
# Filtrage temporel optionnel avant segmentation
if any(PREPROCESS_PROFILE.get(k, False) for k in
       ("apply_temporal_bandpass", "apply_temporal_lowpass", "apply_temporal_highpass")):
    from pipeline.pretraitement import _apply_temporal_filters
    print("Filtrage temporel avant segmentation...")
    filter_pp = {**PREPROCESS_PROFILE, "approach": "B",
                 "subject_id_col": "subject_id", "time_col": "time"}
    raw_df = _apply_temporal_filters(raw_df, feature_cols, filter_pp)
    print("Filtrage termine.")

target_df_raw = load_per_minute_targets(TARGET_PROFILE)
target_df_raw["subject_id"] = target_df_raw["subject_id"].astype(str).str.strip()
target_df_raw = apply_target_discretization(target_df_raw, TARGET_PROFILE)
target_df_raw["subject_id"] = target_df_raw["subject_id"].astype(str)
print("Cibles chargees : " + str(target_df_raw.shape))
print("Classes : " + str(sorted(target_df_raw["target"].dropna().unique().tolist())))
display(target_df_raw.head())

seg_mode = RAW_PROFILE.get("segmentation_mode", "per_minute")
T = RAW_PROFILE["sequence_length"]

if seg_mode == "sliding_window":
    window_s = RAW_PROFILE["window_s"]
    stride_s = RAW_PROFILE["stride_s"]
    print("Segmentation fenetres glissantes : W=" + str(window_s) + "s, S=" + str(stride_s) + "s, T=" + str(T))
    X_raw, y_raw, groups_raw = segment_sequences_sliding_window(
        raw_df=raw_df, feature_cols=feature_cols, target_df=target_df_raw,
        window_s=window_s, stride_s=stride_s, T=T,
        subject_col="subject_id", time_col="time", pad_value=RAW_PROFILE["pad_value"],
    )
else:
    print("Segmentation par minute : T=" + str(T))
    X_raw, y_raw, groups_raw = segment_sequences_by_minute(
        raw_df=raw_df, feature_cols=feature_cols, target_df=target_df_raw,
        T=T, subject_col="subject_id", time_col="time", pad_value=RAW_PROFILE["pad_value"],
    )

print("X shape : " + str(X_raw.shape) + "  (n_segments, T, n_canaux)")
print("y shape : " + str(y_raw.shape))
print("Segments par classe :")
import collections
for k, v in sorted(collections.Counter(y_raw).items()):
    print("  " + str(k) + ": " + str(v))


## 5) Prétraitement 3D, split et préparation

In [ ]:
prepared = prepare_splits_and_impute_3d(
    X=X_raw,
    y=y_raw,
    groups=groups_raw,
    preprocess_profile=PREPROCESS_PROFILE,
    model_profile=MODEL_PROFILE,
)

X_train_imp = prepared["X_train_imp"]
X_val_imp   = prepared["X_val_imp"]
X_test_imp  = prepared["X_test_imp"]
y_train     = prepared["y_train"]
y_val       = prepared["y_val"]
y_test      = prepared["y_test"]
train_idx   = prepared["train_idx"]
val_idx     = prepared["val_idx"]
test_idx    = prepared["test_idx"]
imputer     = prepared["imputer"]
scaler      = prepared["scaler"]

print(f"Split train/val/test : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"X_train shape        : {X_train_imp.shape}")
if scaler is not None:
    print(f"Normalisation        : {type(scaler).__name__}")

## 5.quat) Répartition des canaux par flux

In [ ]:
stream_map = split_feature_streams(feature_cols, FUSION_PROFILE)

for stream_name, cols in stream_map.items():
    print(f"\n--- Flux '{stream_name}' ({len(cols)} canaux) ---")
    print(cols)

total_assigned = sum(len(v) for v in stream_map.values())
print(f"\nTotal canaux assignés : {total_assigned} / {len(feature_cols)}")

## 6) Entraînement des modèles de flux et du méta-modèle

**Protocole sans fuite :**
1. Modèles de flux entraînés sur `train` uniquement
2. Méta-features construites sur `train` + `val`
3. Méta-modèle entraîné sur `train + val`
4. Évaluation finale sur `test`

In [ ]:
print("=== Entraînement des modèles de flux ===")
stream_models = train_stream_models(
    X_train=X_train_imp,
    y_train=y_train,
    X_val=X_val_imp,
    y_val=y_val,
    stream_map=stream_map,
    feature_cols=feature_cols,
    model_profile=STREAM_PROFILES if STREAM_PROFILES is not None else MODEL_PROFILE,
)

is_classif = MODEL_PROFILE["task_type"] == "classification"

print("\n=== Construction des méta-features ===")
meta_X_train = make_meta_features(stream_models, stream_map, X_train_imp, feature_cols, is_classif)
meta_X_val   = make_meta_features(stream_models, stream_map, X_val_imp,   feature_cols, is_classif)
meta_X_test  = make_meta_features(stream_models, stream_map, X_test_imp,  feature_cols, is_classif)
print(f"Shape méta-features train : {meta_X_train.shape}")
print(f"Shape méta-features val   : {meta_X_val.shape}")
print(f"Shape méta-features test  : {meta_X_test.shape}")

print("\n=== Entraînement du méta-modèle ===")
meta_model = build_meta_model(FUSION_PROFILE, MODEL_PROFILE["task_type"], seed=SEED)
meta_model.fit(
    np.vstack([meta_X_train, meta_X_val]),
    np.concatenate([y_train, y_val]),
)
print("Méta-modèle :", type(meta_model).__name__)

## 7) Évaluation, robustesse et visualisations

In [ ]:
pred_test, metrics, classification_text_report = evaluate_test_set(
    final_model=meta_model,
    X_test_imp=meta_X_test,
    y_test=y_test,
    model_profile=MODEL_PROFILE,
    target_profile=TARGET_PROFILE,
    show_plots=True,
)

print(pd.Series(metrics))
if classification_text_report is not None:
    print("\nClassification report:")
    print(classification_text_report)

noise_scores = evaluate_robustness(
    final_model=meta_model,
    X_test_imp=meta_X_test,
    y_test=y_test,
    model_profile=MODEL_PROFILE,
    eval_profile=EVAL_PROFILE,
    seed=SEED,
    show_plot=True,
)
print("Score robustesse (moyenne) :", float(np.mean(noise_scores)))

plot_pca_if_classification(meta_X_test, y_test, MODEL_PROFILE, seed=SEED)

### 7.bis) Performance individuelle par flux

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from pipeline.fusion import _col_indices

print("Performance individuelle de chaque flux sur le test set:\n")
for stream_name, stream_model in stream_models.items():
    idx = _col_indices(stream_map[stream_name], feature_cols)
    if X_test_imp.ndim == 3:
        X_s = X_test_imp[:, :, idx]
        if not hasattr(stream_model, "input_shape"):
            X_s = X_s.reshape(len(X_s), -1)
    else:
        X_s = X_test_imp[:, idx]
        if hasattr(stream_model, "input_shape"):
            X_s = X_s.reshape(-1, X_s.shape[1], 1)
    pred_s = stream_model.predict(X_s)

    if is_classif:
        acc = accuracy_score(y_test, pred_s)
        f1  = f1_score(y_test, pred_s, average="weighted", zero_division=0)
        print(f"  [{stream_name}]  accuracy={acc:.3f}  F1={f1:.3f}")
    else:
        mae  = mean_absolute_error(y_test, pred_s)
        rmse = float(np.sqrt(np.mean((y_test - pred_s) ** 2)))
        print(f"  [{stream_name}]  MAE={mae:.3f}  RMSE={rmse:.3f}")

print()
if is_classif:
    print(f"  [fusion]  accuracy={metrics['accuracy']:.3f}  F1={metrics['f1_weighted']:.3f}")
else:
    print(f"  [fusion]  MAE={metrics['mae']:.3f}  RMSE={metrics['rmse']:.3f}")

## 8) Sauvegarde du modèle et rapport visuel

In [ ]:
import joblib

out_dir = OUTPUT_PROFILE["output_dir"]
os.makedirs(out_dir, exist_ok=True)

joblib.dump(meta_model, os.path.join(out_dir, "meta_model.joblib"))
for name, m in stream_models.items():
    joblib.dump(m, os.path.join(out_dir, f"stream_{name}.joblib"))
if imputer is not None:
    joblib.dump(imputer, os.path.join(out_dir, "imputer.joblib"))
if scaler is not None:
    joblib.dump(scaler, os.path.join(out_dir, "scaler.joblib"))

print("Modèles sauvegardés :")
print("- meta_model.joblib")
for name in stream_models:
    print(f"- stream_{name}.joblib")

# Construire un dataset_df 2D pour le rapport visuel (moyenne temporelle par segment)
X_all_mean = X_raw.mean(axis=1)  # (n_segments, n_features) — moyenne sur T
dataset_df_report = pd.DataFrame(X_all_mean, columns=feature_cols)
dataset_df_report["target"]     = y_raw
dataset_df_report["subject_id"] = groups_raw
dataset_df_report["row_id"]     = np.arange(len(y_raw))

export_visual_report(
    dataset_df=dataset_df_report,
    feature_cols=feature_cols,
    model_profile=MODEL_PROFILE,
    output_profile=OUTPUT_PROFILE,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    target_profile=TARGET_PROFILE,
    raw_df=dataset_df_report,
    preprocess_profile=PREPROCESS_PROFILE,
    metrics=metrics,
    final_model=meta_model,
    X_test_imp=meta_X_test,
    y_test=y_test,
)